# 96 — Autonomous Coding Assistant for One Repo

## Goal
Build an agent that can **search code**, **run tests**,
**propose edits**, and **summarize changes** — a mini
AI coding assistant scoped to a single repository.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Code-aware tools** | Search, read, grep within a codebase |
| **Test execution** | LLM triggers test runs and reads output |
| **Edit proposals** | Agent suggests code changes with diffs |
| **LangGraph workflow** | Multi-step coding workflow |

## Stack
- `langgraph` — stateful coding workflow
- `langchain-ollama` — local LLM
- Ollama `qwen3.5:9b`

> **Note:** Uses a simulated repo filesystem. In production,
> connect to real file I/O and subprocess for test execution.

In [ ]:
import os, warnings, json, re
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Simulated Repository

In [ ]:
# Simulated codebase: a small Python project
REPO = {
    "src/calculator.py": '''"""Simple calculator module."""

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b  # BUG: no zero division check
''',
    "src/utils.py": '''"""Utility functions."""

def format_result(value):
    """Format a number for display."""
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)

def validate_input(x):
    """Check if input is numeric."""
    try:
        float(x)
        return True
    except (TypeError, ValueError):
        return False
''',
    "tests/test_calculator.py": '''"""Tests for calculator."""
import sys; sys.path.insert(0, "src")
from calculator import add, subtract, multiply, divide

def test_add():
    assert add(2, 3) == 5

def test_subtract():
    assert subtract(5, 3) == 2

def test_multiply():
    assert multiply(4, 3) == 12

def test_divide():
    assert divide(10, 2) == 5.0

def test_divide_by_zero():
    # This test should fail due to the bug
    try:
        divide(1, 0)
        assert False, "Should have raised an error"
    except ZeroDivisionError:
        pass  # Expected
''',
    "README.md": "# Calculator Project\nA simple calculator with basic operations."
}

print(f"Repo files: {list(REPO.keys())}")

In [ ]:
# Coding tools
def tool_search_code(pattern: str) -> str:
    """Search for a pattern across all files in the repo."""
    matches = []
    for path, content in REPO.items():
        for i, line in enumerate(content.split("\n"), 1):
            if pattern.lower() in line.lower():
                matches.append(f"{path}:{i}: {line.strip()}")
    return "\n".join(matches) if matches else f"No matches for '{pattern}'"

def tool_read_file(path: str) -> str:
    """Read a file from the repo."""
    if path in REPO:
        return REPO[path]
    return f"ERROR: File '{path}' not found"

def tool_list_files() -> str:
    """List all files in the repo."""
    return "\n".join(REPO.keys())

def tool_run_tests() -> str:
    """Simulate running the test suite."""
    # Simulate test results based on the current code
    has_zero_check = "b == 0" in REPO.get("src/calculator.py", "") or \
                     "b != 0" in REPO.get("src/calculator.py", "")
    results = [
        "test_add ........................ PASSED",
        "test_subtract ................... PASSED",
        "test_multiply ................... PASSED",
        "test_divide ..................... PASSED",
    ]
    if has_zero_check:
        results.append("test_divide_by_zero ............. PASSED")
        results.append("\n5 passed, 0 failed")
    else:
        results.append("test_divide_by_zero ............. FAILED")
        results.append("  ZeroDivisionError: division by zero")
        results.append("\n4 passed, 1 failed")
    return "\n".join(results)

CODING_TOOLS = {
    "search_code": tool_search_code,
    "read_file": tool_read_file,
    "list_files": tool_list_files,
    "run_tests": tool_run_tests
}

print(f"Coding tools: {list(CODING_TOOLS.keys())}")

## Step 2 — Coding Agent Graph

```
understand_task → explore_code → run_tests → propose_fix → summarize → END
```

In [ ]:
class CodingState(TypedDict):
    task: str
    file_listing: str
    relevant_code: str
    test_results: str
    proposed_fix: str
    summary: str

def explore_code(state: CodingState) -> CodingState:
    """Explore the codebase to understand structure."""
    print("\n🔍 Exploring codebase...")
    listing = tool_list_files()
    print(f"   Files: {listing}")
    
    # Search for relevant code based on the task
    search_results = tool_search_code("divide")
    print(f"   Search 'divide': {len(search_results.split(chr(10)))} matches")
    
    # Read the main module
    main_code = tool_read_file("src/calculator.py")
    test_code = tool_read_file("tests/test_calculator.py")
    
    relevant = f"=== src/calculator.py ===\n{main_code}\n\n=== tests/test_calculator.py ===\n{test_code}"
    
    return {**state, "file_listing": listing, "relevant_code": relevant}

def run_tests(state: CodingState) -> CodingState:
    """Run the test suite."""
    print("\n🧪 Running tests...")
    results = tool_run_tests()
    print(results)
    return {**state, "test_results": results}

print("Nodes: explore_code, run_tests defined")

In [ ]:
def propose_fix(state: CodingState) -> CodingState:
    """LLM proposes a code fix based on test failures."""
    print("\n💡 Proposing fix...")
    
    msg = llm.invoke([
        SystemMessage(content="""You are a senior Python developer. Analyze the failing tests
and propose a fix.

Return your response as:
1. DIAGNOSIS: What's wrong (1-2 sentences)
2. FIX: The corrected code for the function that needs fixing
3. DIFF: Show the change in unified diff format

Do NOT use thinking tags."""),
        HumanMessage(content=f"Task: {state['task']}\n\nCode:\n{state['relevant_code']}\n\n"
                             f"Test Results:\n{state['test_results']}")
    ])
    
    fix = msg.content
    if "<think>" in fix:
        fix = fix.split("</think>")[-1].strip()
    
    print(f"   Fix proposed ({len(fix)} chars)")
    return {**state, "proposed_fix": fix}

def summarize_changes(state: CodingState) -> CodingState:
    """Generate a concise summary of the investigation and fix."""
    print("\n📋 Summarizing...")
    
    msg = llm.invoke([
        SystemMessage(content="""Write a brief PR-style summary (100-150 words) of
what was found and fixed. Include:
- What the bug was
- What the fix does
- Test impact
Do NOT use thinking tags."""),
        HumanMessage(content=f"Fix:\n{state['proposed_fix']}\n\nTests:\n{state['test_results']}")
    ])
    
    summary = msg.content
    if "<think>" in summary:
        summary = summary.split("</think>")[-1].strip()
    
    return {**state, "summary": summary}

print("Nodes: propose_fix, summarize_changes defined")

In [ ]:
# Build the graph
graph = StateGraph(CodingState)

graph.add_node("explore", explore_code)
graph.add_node("test", run_tests)
graph.add_node("fix", propose_fix)
graph.add_node("summarize", summarize_changes)

graph.set_entry_point("explore")
graph.add_edge("explore", "test")
graph.add_edge("test", "fix")
graph.add_edge("fix", "summarize")
graph.add_edge("summarize", END)

coding_app = graph.compile()
print("Coding assistant graph compiled!")

In [ ]:
# Run the coding assistant
result = coding_app.invoke({
    "task": "Find and fix the bug in the calculator divide function that causes test_divide_by_zero to fail",
    "file_listing": "",
    "relevant_code": "",
    "test_results": "",
    "proposed_fix": "",
    "summary": ""
})

print("\n" + "=" * 60)
print("PROPOSED FIX")
print("=" * 60)
print(result["proposed_fix"])

print("\n" + "=" * 60)
print("PR SUMMARY")
print("=" * 60)
print(result["summary"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Code search** | Pattern matching across repo files |
| **Test execution** | Simulated pytest with pass/fail reporting |
| **Fix proposal** | LLM generates diagnosis + diff |
| **Change summary** | PR-style description of the fix |

## Coding Agent Workflow
```
Task → Explore (list, search, read) → Run Tests
  → Propose Fix (LLM) → Summarize Changes → END
```

## Extending This Project
- Real file I/O instead of simulated repo
- `subprocess.run(['pytest'])` for actual test execution
- Auto-apply fixes and re-run tests to verify
- Git integration (create branch, commit, PR)
- Multi-file fixes via iterative exploration